In [ ]:
# !pip install unsloth trl peft accelerate bitsandbytes

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
import torch
import torch.nn as nn
from datasets import load_dataset
from unsloth import FastLanguageModel

In [ ]:
model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"

max_seq_length = 2048
dtype = None

#load model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True
    )

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_dropout = 0,
    lora_alpha=16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = True
)

In [ ]:
dataset =load_dataset("tatsu-lab/alpaca")
dataset

In [ ]:
# Randomly sample 10,000 examples
dataset = dataset["train"].shuffle(seed=42).select(range(10000))
dataset

In [ ]:
split_dataset = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

In [ ]:
len(train_dataset)

In [ ]:
train_dataset.column_names

In [ ]:
train_dataset[0]

In [ ]:
# text = train_dataset["text"]
# text[0]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="qwen-2.5",   # or "chatml" depending on your Unsloth version
)

def formatting_prompts_func(examples):
    texts = []

    for instruction, input_text, output in zip(
        examples["instruction"],
        examples["input"],
        examples["output"],
    ):
        user_message = instruction
        if input_text.strip():
            user_message += "\n\n" + input_text

        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output},
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        texts.append(text)

    return {"text": texts}

In [ ]:
train_dataset = train_dataset.map(
    formatting_prompts_func,
    batched=True,
)

eval_dataset = eval_dataset.map(
    formatting_prompts_func,
    batched=True,
)

In [ ]:
print(train_dataset)
print(train_dataset[0])

In [ ]:
from transformers import TrainingArguments

In [ ]:
args = TrainingArguments(
    output_dir = "results",
    learning_rate = 1e-4,
    num_train_epochs = 2,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "no",
    fp16 = True,
    bf16 = False,
    push_to_hub = True,
    hub_model_id = "ciphermosaic/qwen-alpaca-lora"
)

In [ ]:
print(train_dataset[0])

In [ ]:
from trl import SFTTrainer

In [ ]:
import trl
print(trl.__version__)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args = args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length
)

In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

In [ ]:
print(dataset)
print(train_dataset)
print(train_dataset.column_names)
print(train_dataset[0])

In [ ]:
trainer.train()

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

print(pipe(
    "Give me five interview tips for a software engineer.",
    max_new_tokens = 100)[0]["generated_text"]
)

Save model

In [ ]:
model.save_pretrained("qwen-alpaca-lora")
tokenizer.save_pretrained("qwen-alpaca-lora")

push model

In [ ]:
model.push_to_hub("ciphermosaic/qwen-alpaca-lora")
tokenizer.push_to_hub("ciphermosaic/qwen-alpaca-lora")

merging

In [ ]:
model.save_pretrained_merged(
    "qwen-alpaca-merged",
    tokenizer,
    save_method = "merged_16bit"
    )

push merged model

In [ ]:
model.push_to_hub_merged(
    "ciphermosaic/qwen-alpaca-merged",
    tokenizer,
    save_method="merged_16bit",
)

convert to gguf

In [ ]:
model.save_pretrained_gguf(
    "qwen-alpaca-gguf",
    tokenizer,
    quantization_method="q4_k_m"
    )

push gguf to hub

In [ ]:
model.push_to_hub_gguf(
    "ciphermosaic/qwen-alpaca-gguf",
    tokenizer,
    quantization_method="q4_k_m"
)